In [ ]:
import pandas as pd

# Read in dataset from data preprocessing
df = pd.read_csv("clean_matches.csv")
df

In [ ]:
# Function that builds the search query that will be input to Google
import calendar

def build_search_query(match):                              # Match will be row from the dataset
    # Set up variables that will be used in the string for search query
    home_team = match["home team"]
    away_team = match["away team"]
    month = calendar.month_name[int(match["date"][5:7])]    # Calendar converts the numbers 1-12 to the corresponding months January-December
    year = match["date"][0:4]
    return f"{home_team} vs {away_team} in {month} {year}"  # Return the search query in an f-string

In [ ]:
# Test function from previous cell
build_search_query(df.iloc[44])

In [ ]:
# Add the seach query as a column in the database
df["search query"] = df.apply(build_search_query, axis=1)
df

In [ ]:
# API key needs to be kept private from public GITHUB repository
# Key stored in an .env file
# .env file listed within .gitignore
# This cell reads in the key and stores it with api_key

from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("SERPAPI_KEY")

In [ ]:
# Function that performs the Google search query
# Uses SerpAPI

from serpapi import GoogleSearch

def get_urls(query):
    # Parameters for the Google search
    params = {
        "q": query,                     # The string to be query                
        "api_key": api_key,             # The SerpAPI key
        "num": 20                       # Max 20 links, but will naturally restrict to the first page of Google searches
    }

    search = GoogleSearch(params)       
    results = search.get_dict()         # Sends request to the API

    return [r["link"] for r in results.get("organic_results", [])]          # Extract links

# Test the function get_urls on the query generated for one of the matches
# Need to be careful about overusing this function: get 250 queries per month through free tier of SerpAPI

urls = get_urls("Man United vs Birmingham in January 2008")
urls

In [ ]:
# Access the urls variable from the previous cell without needing to send another query to SERPAPI
urls

In [ ]:
# Able to redefine urls from above cell if I start a new session without needing to re-use a SERPAPI search
urls = ['https://www.youtube.com/watch?v=givC4ZvbqYc',
 'https://www.skysports.com/football/manchester-united-vs-birmingham-city/teams/94538',
 'https://www.transfermarkt.us/manchester-united_birmingham-city/index/spielbericht/81687',
 'https://www.manutd.com/en/videos/detail/man-utd-1-birmingham-city-0-extended-highlights-premier-league-2007-08',
 'https://www.statmuse.com/fc/ask/full-time-scores-of-man-u-vs-birmingham-in-premiere-league-match-in-2008',
 'http://news.bbc.co.uk/sport2/hi/football/eng_prem/7163892.stm',
 'https://www.premierleague.com/en/match/128642/manchester-united-vs-birmingham-city',
 'https://www.espn.ph/football/report/_/gameId/220076',
 'https://www.footballcritic.com/premier-league-manchester-united-fc-birmingham-city-fc/match-stats/42194']

In [ ]:
# Test on one match before expanding to all to preserve SerpAPI queries

sample_df = df.iloc[[1]].copy()        # Make a copy of the second entry in a sample dataframe
sample_df["article urls"] = [urls]     # Add a new column of the list of url strings
sample_df